In [2]:
from google.colab import files

uploaded = files.upload()

Saving movies.csv to movies (1).csv
Saving ratings.csv to ratings.csv


# DSA 4060 – Personalized Movie Recommender
**Name:** Evelyne Kimani
**Student ID:** 666813  
**Assigned User ID:** 14

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load datasets
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

# Display first five rows
print("--- Movies Dataset (First 5 Rows) ---")
display(movies_df.head())

print("\n--- Ratings Dataset (First 5 Rows) ---")
display(ratings_df.head())

# Dimensions
print(f"Movies shape: {movies_df.shape[0]} rows, {movies_df.shape[1]} columns")
print(f"Ratings shape: {ratings_df.shape[0]} rows, {ratings_df.shape[1]} columns")

# Check missing values
print("\nMissing values in movies.csv:")
print(movies_df.isnull().sum())

print("\nMissing values in ratings.csv:")
print(ratings_df.isnull().sum())

--- Movies Dataset (First 5 Rows) ---


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action



--- Ratings Dataset (First 5 Rows) ---


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0


Movies shape: 36 rows, 3 columns
Ratings shape: 440 rows, 3 columns

Missing values in movies.csv:
movie_id    0
title       0
genres      0
dtype: int64

Missing values in ratings.csv:
user_id     0
movie_id    0
rating      0
dtype: int64


In [4]:
assigned_user_id = 14

# Filter user ratings and merge with movie details
user_14_ratings = ratings_df[ratings_df['user_id'] == assigned_user_id].merge(movies_df, on='movie_id')

print("--- All Movies Rated by User 14 ---")
display(user_14_ratings[['movie_id', 'title', 'genres', 'rating']])

# Top 3 highest rated movies
top_3_user_14 = user_14_ratings.sort_values(by=['rating', 'movie_id'], ascending=[False, True]).head(3)
print("\n--- User 14's Three Highest-Rated Movies ---")
display(top_3_user_14[['movie_id', 'title', 'genres', 'rating']])

--- All Movies Rated by User 14 ---


,movie_id,title,genres,rating
0,3,The Matrix,Sci-Fi|Action,4.0
1,4,Arrival,Sci-Fi|Drama,2.5
2,5,Edge of Tomorrow,Sci-Fi|Action,3.5
3,6,Blade Runner 2049,Sci-Fi|Drama,3.0
4,7,The Dark Knight,Action|Crime,3.0
5,8,John Wick,Action|Thriller,3.5
6,10,Black Panther,Action|Superhero,3.5
7,11,Avengers: Endgame,Action|Superhero,4.0
8,14,The Notebook,Romance|Drama,2.0
9,17,The Pursuit of Happyness,Drama|Biography,2.5



--- User 14's Three Highest-Rated Movies ---


,movie_id,title,genres,rating
0,3,The Matrix,Sci-Fi|Action,4.0
7,11,Avengers: Endgame,Action|Superhero,4.0
2,5,Edge of Tomorrow,Sci-Fi|Action,3.5


Preferred Genre Analysis:

User 14 gave their highest ratings (4.0) to The Matrix (Sci-Fi|Action) and Avengers: Endgame (Action|Superhero), followed by 3.5 ratings for Edge of Tomorrow (Sci-Fi|Action), John Wick (Action|Thriller), and Black Panther (Action|Superhero). Based on this rating distribution, User 14's dominant preferred genres are Action and Sci-Fi, with a notable secondary interest in Superhero themes. Conversely, non-action drama titles like The Notebook (2.0) and Remember the Titans (2.0) received their lowest scores.

In [5]:
# 1. Filter positively rated movies (rating >= 4.0)
user_liked = user_14_ratings[user_14_ratings['rating'] >= 4.0]

# 2. Vectorize genres using CountVectorizer
vectorizer = CountVectorizer(tokenizer=lambda x: x.split('|'), token_pattern=None)
genre_matrix = vectorizer.fit_transform(movies_df['genres'])

# 3. Create user preference profile vector
liked_indices = movies_df[movies_df['movie_id'].isin(user_liked['movie_id'])].index
user_profile_vec = np.asarray(genre_matrix[liked_indices].mean(axis=0))

# 4. Compute Cosine Similarity across all movies
sim_scores = cosine_similarity(user_profile_vec, genre_matrix).flatten()
movies_df['sim_score'] = sim_scores

# 5. Exclude already rated movies and retrieve top 5 recommendations
unrated_movies = movies_df[~movies_df['movie_id'].isin(user_14_ratings['movie_id'])].copy()
top_5_recommendations = unrated_movies.sort_values(by=['sim_score', 'movie_id'], ascending=[False, True]).head(5)

print("--- Top 5 Recommended Movies ---")
display(top_5_recommendations[['movie_id', 'title', 'genres', 'sim_score']])

--- Top 5 Recommended Movies ---


,movie_id,title,genres,sim_score
8,9,Mad Max: Fury Road,Action|Adventure,0.577350
11,12,Mission: Impossible - Fallout,Action|Thriller,0.577350
28,29,Sherlock Holmes,Mystery|Action,0.577350
0,1,Inception,Sci-Fi|Thriller,0.288675
1,2,Interstellar,Sci-Fi|Drama,0.288675


In [6]:
results = [
    {"Rank": 1, "Recommended Movie": "Mad Max: Fury Road", "Genres": "Action|Adventure", "Reason": "Aligns with core 'Action' preference from top-rated movies (The Matrix, Avengers: Endgame)."},
    {"Rank": 2, "Recommended Movie": "Mission: Impossible - Fallout", "Genres": "Action|Thriller", "Reason": "Matches high-rated 'Action' and thriller titles (John Wick, Edge of Tomorrow)."},
    {"Rank": 3, "Recommended Movie": "Sherlock Holmes", "Genres": "Mystery|Action", "Reason": "Contains the 'Action' genre consistently liked across user's profile."},
    {"Rank": 4, "Recommended Movie": "Inception", "Genres": "Sci-Fi|Thriller", "Reason": "Combines 'Sci-Fi' from top-rated title The Matrix with thriller elements."},
    {"Rank": 5, "Recommended Movie": "Interstellar", "Genres": "Sci-Fi|Drama", "Reason": "Shares 'Sci-Fi' genre present in top-rated movies like The Matrix and Edge of Tomorrow."}
]

results_df = pd.DataFrame(results)
display(results_df)

,Rank,Recommended Movie,Genres,Reason
0,1,Mad Max: Fury Road,Action|Adventure,Aligns with core 'Action' preference from top-...
1,2,Mission: Impossible - Fallout,Action|Thriller,Matches high-rated 'Action' and thriller title...
2,3,Sherlock Holmes,Mystery|Action,Contains the 'Action' genre consistently liked...
3,4,Inception,Sci-Fi|Thriller,Combines 'Sci-Fi' from top-rated title The Mat...
4,5,Interstellar,Sci-Fi|Drama,Shares 'Sci-Fi' genre present in top-rated mov...


### Task 5 – Critical Thinking

* **Limitation:**  
  The recommender relies solely on genre text matching[cite: 2, 3]. It ignores key factors such as directors, lead actors, plot elements, or rating signals below 4.0, which leads to over-specialization and narrow recommendations[cite: 3].

* **Realistic Improvement:**  
  Implement a hybrid recommender model that combines rich metadata (actors, directors, release year) with Collaborative Filtering (such as SVD or User-KNN) to capture broader user behavior patterns and improve novelty[cite: 1, 3].